In [6]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Customer_support_chatbot'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from datasets import load_dataset

support_ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
print(support_ds)

df = support_ds['train'].to_pandas()
df.head()

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [12]:
print(df['intent'].unique())

['cancel_order' 'change_order' 'change_shipping_address'
 'check_cancellation_fee' 'check_invoice' 'check_payment_methods'
 'check_refund_policy' 'complaint' 'contact_customer_service'
 'contact_human_agent' 'create_account' 'delete_account'
 'delivery_options' 'delivery_period' 'edit_account' 'get_invoice'
 'get_refund' 'newsletter_subscription' 'payment_issue' 'place_order'
 'recover_password' 'registration_problems' 'review'
 'set_up_shipping_address' 'switch_account' 'track_order' 'track_refund']


In [15]:
intent_map = {
    # order_status
    'track_order': 'order_status',
    'delivery_options': 'order_status',
    'delivery_period': 'order_status',

    # order_management
    'cancel_order': 'order_management',
    'change_order': 'order_management',
    'place_order': 'order_management',

    # billing_and_refunds
    'check_invoice': 'billing_and_refunds',
    'get_invoice': 'billing_and_refunds',
    'get_refund': 'billing_and_refunds',
    'track_refund': 'billing_and_refunds',
    'payment_issue': 'billing_and_refunds',
    'check_payment_methods': 'billing_and_refunds',
    'check_refund_policy': 'billing_and_refunds',
    'check_cancellation_fee': 'billing_and_refunds',

     # account_management
    'create_account': 'account_management',
    'edit_account': 'account_management',
    'delete_account': 'account_management',
    'switch_account': 'account_management',
    'recover_password': 'account_management',
    'registration_problems': 'account_management',
    'change_shipping_address': 'account_management',
    'set_up_shipping_address': 'account_management',

    # complaint
    'complaint': 'complaint',
    'review': 'complaint',
    'contact_customer_service': 'complaint',
    'contact_human_agent': 'complaint',

    # out_of_scope (catch-all for anything left)
    'newsletter_subscription': 'out_of_scope',
}

df['intent_bucket'] = df['intent'].map(intent_map)
print(df['intent_bucket'].value_counts(dropna=False))

intent_bucket
account_management     7956
billing_and_refunds    7939
complaint              3996
order_management       2993
order_status           2989
out_of_scope            999
Name: count, dtype: int64


Data Splitting

In [17]:
# split yourself - this dataset has no pre-made val/test split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2,
                                     random_state=42,
                                     stratify=df['intent_bucket'])
val_df, test_df = train_test_split(temp_df,
                                   test_size=0.5,
                                   random_state=42,
                                   stratify=temp_df['intent_bucket'])

print(len(train_df), len(val_df), len(test_df))

21497 2687 2688


vectorize + train

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

vectorizer = TfidfVectorizer(max_features=5000,
                             ngram_range=(1,2))
X_train = vectorizer.fit_transform(train_df['instruction'])
X_val = vectorizer.transform(val_df['instruction'])
X_test = vectorizer.transform(test_df['instruction'])

y_train = train_df['intent_bucket']
y_val = val_df['intent_bucket']
y_test = test_df['intent_bucket']

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

val_preds = clf.predict(X_val)
print("Accuracy Score:", accuracy_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

Accuracy Score: 0.9962783773725344
                     precision    recall  f1-score   support

 account_management       0.99      1.00      1.00       795
billing_and_refunds       1.00      1.00      1.00       794
          complaint       1.00      1.00      1.00       399
   order_management       1.00      0.99      0.99       300
       order_status       1.00      1.00      1.00       299
       out_of_scope       1.00      0.96      0.98       100

           accuracy                           1.00      2687
          macro avg       1.00      0.99      0.99      2687
       weighted avg       1.00      1.00      1.00      2687



In [20]:
test_preds = clf.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, test_preds))

import joblib, os
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
joblib.dump(clf, f'{PROJECT_DIR}/models/intent_classifier.pkl')
joblib.dump(vectorizer, f'{PROJECT_DIR}/models/intent_vectorizer.pkl')
joblib.dump(intent_map, f'{PROJECT_DIR}/models/intent_map.pkl')

Test accuracy: 0.9962797619047619


['/content/drive/MyDrive/Customer_support_chatbot/models/intent_map.pkl']

Quick sanity check

In [21]:
def detect_intent(text):
    vec = vectorizer.transform([text])
    return clf.predict(vec)[0]

print(detect_intent("Where is my package?"))          # expect order_status
print(detect_intent("I want a refund for my order"))  # expect billing_and_refunds
print(detect_intent("This service is terrible"))      # expect complaint

order_status
billing_and_refunds
complaint
